In [1]:
# Import the 'drive' module from the 'google.colab' package to enable Google Drive integration.
# Then, mount Google Drive to the '/drive' directory within the Colab environment.
# The 'force_remount=True' parameter ensures that the Drive is remounted even if it was previously mounted.

from google.colab import drive
drive.mount('/drive', force_remount=True)

# Change the current working directory to the specified folder within Google Drive,
# where you can save and load your Colab notebooks or files.
%cd '/drive/MyDrive/Colab Notebooks/Employee Communications'

Mounted at /drive
/drive/MyDrive/Colab Notebooks/Employee Communications


In [2]:
# Importing Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Data Preprocessing

1. Load and inspect the database

In [3]:
# Load the dataset from the Google Drive
df = pd.read_csv("Communications Data.csv",thousands=",",parse_dates=["Publish Date"],
                 header=0)
df.head()

,Comm ID,Title,Publish Date,Day of Week,Post Type,Topic,Sender,Target Audience Size,Unique Viewers,Total Views,Reactions,Comments,Shares,Link Included,Link Clicks
0,C101,Team offsite photo recap,2026-09-03,Thursday,Standard Post,Culture,People Leader,520,383,463,79,7,0,No,NaN
1,C102,Learning week: sessions you can join,2026-09-04,Friday,Standard Post,Learning,HR,18000,4706,6689,270,22,8,Yes,425.0
2,C103,Coffee chat series returns,2026-09-07,Monday,Standard Post,Culture,Comms Team,18000,3222,4403,170,14,8,No,NaN
3,C104,Customer story: helping a small business grow,2026-09-11,Friday,Standard Post,Business Update,Comms Team,18000,3247,4195,139,20,6,No,NaN
4,C105,Meet the 2026 Community Volunteer Award winners,2026-09-14,Monday,Standard Post,Recognition,Comms Team,18000,1991,2702,100,12,11,No,NaN


In [4]:
# Inspect the data types of the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Comm ID               43 non-null     object        
 1   Title                 43 non-null     object        
 2   Publish Date          43 non-null     datetime64[ns]
 3   Day of Week           43 non-null     object        
 4   Post Type             43 non-null     object        
 5   Topic                 43 non-null     object        
 6   Sender                43 non-null     object        
 7   Target Audience Size  43 non-null     int64         
 8   Unique Viewers        43 non-null     int64         
 9   Total Views           43 non-null     int64         
 10  Reactions             43 non-null     int64         
 11  Comments              43 non-null     int64         
 12  Shares                43 non-null     int64         
 13  Link Included         

2. Check the missing values

In [5]:
df.isnull().sum()

,0
Comm ID,0
Title,0
Publish Date,0
Day of Week,0
Post Type,0
Topic,0
Sender,0
Target Audience Size,0
Unique Viewers,0
Total Views,0


All 15 missing values in Link Clicks correspond to posts without links. According to the data dictionary, these are expected structural missing values rather than data quality issues. Therefore, they are retained as missing and excluded from click-related calculations where appropriate.

3. Check for duplicate rows

In [6]:
duplicates = df[df.duplicated(keep=False)]
duplicates

,Comm ID,Title,Publish Date,Day of Week,Post Type,Topic,Sender,Target Audience Size,Unique Viewers,Total Views,Reactions,Comments,Shares,Link Included,Link Clicks
8,C109,Hybrid work guidelines refreshed,2026-09-21,Monday,Standard Post,Workplace,HR,18000,3186,4221,142,13,10,Yes,303.0
9,C109,Hybrid work guidelines refreshed,2026-09-21,Monday,Standard Post,Workplace,HR,18000,3186,4221,142,13,10,Yes,303.0


Drop the duplicate row, since it's exact duplicate

In [7]:
df = df.drop_duplicates()
df

,Comm ID,Title,Publish Date,Day of Week,Post Type,Topic,Sender,Target Audience Size,Unique Viewers,Total Views,Reactions,Comments,Shares,Link Included,Link Clicks
0,C101,Team offsite photo recap,2026-09-03,Thursday,Standard Post,Culture,People Leader,520,383,463,79,7,0,No,NaN
1,C102,Learning week: sessions you can join,2026-09-04,Friday,Standard Post,Learning,HR,18000,4706,6689,270,22,8,Yes,425.0
2,C103,Coffee chat series returns,2026-09-07,Monday,Standard Post,Culture,Comms Team,18000,3222,4403,170,14,8,No,NaN
3,C104,Customer story: helping a small business grow,2026-09-11,Friday,Standard Post,Business Update,Comms Team,18000,3247,4195,139,20,6,No,NaN
4,C105,Meet the 2026 Community Volunteer Award winners,2026-09-14,Monday,Standard Post,Recognition,Comms Team,18000,1991,2702,100,12,11,No,NaN
5,C106,Strategy update: message from our CEO,2026-09-15,Tuesday,Announcement,Business Update,Executive,18000,12239,16257,422,39,54,Yes,966.0
6,C107,Quarterly fraud trends: what employees should ...,2026-09-16,Wednesday,Standard Post,Technology,Comms Team,18000,2036,2722,110,7,5,Yes,153.0
7,C108,Planned system outage this weekend,2026-09-17,Thursday,Standard Post,Technology,IT,18000,4068,5635,219,25,15,No,NaN
8,C109,Hybrid work guidelines refreshed,2026-09-21,Monday,Standard Post,Workplace,HR,18000,3186,4221,142,13,10,Yes,303.0
10,C110,Mental health resources for the season,2026-09-25,Friday,Standard Post,Wellbeing,HR,18000,2272,2714,108,16,9,Yes,142.0


# Exploratory Data Analysis

In [8]:
# Numerical Descriptive Analysis
df[["Target Audience Size", "Unique Viewers", "Total Views", "Reactions", "Comments", "Shares", "Link Clicks"]].describe().round(2)

,Target Audience Size,Unique Viewers,Total Views,Reactions,Comments,Shares,Link Clicks
count,42.00,42.00,42.00,42.00,42.00,42.00,27.0
mean,14429.76,4389.43,5832.24,193.02,26.40,16.02,387.3
std,6557.87,3646.42,4844.88,136.77,46.54,16.60,342.0
min,380.00,211.00,270.00,17.00,1.00,0.00,46.0
25%,18000.00,2068.25,2705.00,110.25,12.00,5.25,159.0
50%,18000.00,3234.50,4312.00,149.50,18.50,10.00,303.0
75%,18000.00,4715.00,6630.50,252.75,24.50,20.25,457.0
max,18000.00,12780.00,18028.00,811.00,312.00,67.00,1549.0


In [9]:
# Categorical Descriptive Analysis
categorical_cols = [
    "Post Type",
    "Topic",
    "Sender",
    "Day of Week",
    "Link Included"
]

for col in categorical_cols:
  print(f"\n{col}")
  display(df[col].value_counts().rename("Posts").to_frame())


Post Type


,Posts
Post Type,
Standard Post,34
Announcement,8



Topic


,Posts
Topic,
Business Update,9
Culture,7
Technology,6
Recognition,5
HR / Benefits,5
Learning,3
Workplace,3
Wellbeing,2
Community,2



Sender


,Posts
Sender,
HR,12
Comms Team,12
Executive,8
IT,5
People Leader,3
Facilities,2



Day of Week


,Posts
Day of Week,
Friday,12
Thursday,10
Monday,7
Wednesday,7
Tuesday,6



Link Included


,Posts
Link Included,
Yes,27
No,15


# Feature Engineering

In [10]:
# Create four key metrics
df["Reach Rate"] = (
    df["Unique Viewers"] / df["Target Audience Size"]
)

df["Engagement Events"] = (
    df["Reactions"] + df["Comments"] + df["Shares"]
)

df["Engagement per Viewer"] = (
    df["Engagement Events"] /
    df["Unique Viewers"].replace(0, np.nan)
)

df["Clicks per Viewer"] = np.where(
    df["Link Included"] == "Yes",
    df["Link Clicks"] /
    df["Unique Viewers"].replace(0, np.nan),
    np.nan)

df.head()

,Comm ID,Title,Publish Date,Day of Week,Post Type,Topic,Sender,Target Audience Size,Unique Viewers,Total Views,Reactions,Comments,Shares,Link Included,Link Clicks,Reach Rate,Engagement Events,Engagement per Viewer,Clicks per Viewer
0,C101,Team offsite photo recap,2026-09-03,Thursday,Standard Post,Culture,People Leader,520,383,463,79,7,0,No,NaN,0.736538,86,0.224543,NaN
1,C102,Learning week: sessions you can join,2026-09-04,Friday,Standard Post,Learning,HR,18000,4706,6689,270,22,8,Yes,425.0,0.261444,300,0.063748,0.09031
2,C103,Coffee chat series returns,2026-09-07,Monday,Standard Post,Culture,Comms Team,18000,3222,4403,170,14,8,No,NaN,0.179000,192,0.059590,NaN
3,C104,Customer story: helping a small business grow,2026-09-11,Friday,Standard Post,Business Update,Comms Team,18000,3247,4195,139,20,6,No,NaN,0.180389,165,0.050816,NaN
4,C105,Meet the 2026 Community Volunteer Award winners,2026-09-14,Monday,Standard Post,Recognition,Comms Team,18000,1991,2702,100,12,11,No,NaN,0.110611,123,0.061778,NaN


In [11]:
post_type_summary = df.groupby("Post Type").agg(
    Posts=("Comm ID", "count"),
    Mean_Reach=("Reach Rate", "mean"),
    Median_Reach=("Reach Rate", "median"),
    Mean_Engagement_Per_Viewer=("Engagement per Viewer", "mean"),
    Mean_Clicks_Per_Viewer=("Clicks per Viewer", "mean")
)
display(post_type_summary.round(3))

print("Lowest reach examples:")
display(df.nsmallest(5, "Reach Rate")[[
    "Comm ID", "Title", "Post Type", "Topic", "Reach Rate"
]])

print("Highest reach examples:")
display(df.nlargest(5, "Reach Rate")[[
    "Comm ID", "Title", "Post Type", "Topic", "Reach Rate"
]])

,Posts,Mean_Reach,Median_Reach,Mean_Engagement_Per_Viewer,Mean_Clicks_Per_Viewer
Post Type,,,,,
Announcement,8,0.647,0.659,0.032,0.063
Standard Post,34,0.270,0.230,0.081,0.083


Lowest reach examples:


,Comm ID,Title,Post Type,Topic,Reach Rate
4,C105,Meet the 2026 Community Volunteer Award winners,Standard Post,Recognition,0.110611
14,C114,Lunch & learn: managing your money,Standard Post,Learning,0.112278
6,C107,Quarterly fraud trends: what employees should ...,Standard Post,Technology,0.113111
15,C115,Volunteer day sign-ups open,Standard Post,Community,0.120278
10,C110,Mental health resources for the season,Standard Post,Wellbeing,0.126222


Highest reach examples:


,Comm ID,Title,Post Type,Topic,Reach Rate
0,C101,Team offsite photo recap,Standard Post,Culture,0.736538
33,C133,Thank you from the Risk team leadership,Standard Post,Recognition,0.720000
17,C117,Benefits enrolment is now open,Announcement,HR / Benefits,0.710000
21,C121,Cybersecurity Awareness Month: new training re...,Announcement,Technology,0.696833
41,C141,Holiday giving campaign launches,Announcement,Community,0.685611


In [12]:
# Export dataset for further analysis
df.to_excel("Employee_Comms_Cleaned.xlsx", index=False)